# ReelForge - Generer de vrais plans video IA (gratuit)

Ce notebook genere de **vraies videos IA** (des scenes qui bougent, pas des images fixes)
sur le **GPU gratuit de Google Colab**.

**Comment l'utiliser**
1. Menu `Execution` -> `Modifier le type d'execution` -> choisir **T4 GPU** (gratuit).
2. Execute les cellules dans l'ordre (`Ctrl+F9` pour tout lancer).
3. Modifie les `PROMPTS` dans la cellule 3.
4. A la fin, telecharge `clips_ia.zip`.
5. Dans ReelForge, depose ces clips dans le champ **"Clips IA generes"** : ils remplaceront
   les visuels automatiques.

> Modele utilise : **Wan 2.1 (1.3B)**, licence Apache 2.0 : 100 % gratuit et commercialisable.
> Chaque clip fait ~5 secondes en 480p. Compte ~2 a 4 minutes par clip sur un T4 gratuit.


In [ ]:
# 1) Verifier que le GPU est bien disponible
!nvidia-smi
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "AUCUN GPU - active le T4 dans Execution > Modifier le type d'execution")


In [ ]:
# 2) Installer les dependances (2-3 minutes)
!pip install -q --upgrade "diffusers>=0.32" transformers accelerate imageio imageio-ffmpeg ftfy


In [ ]:
# 3) Parametres : modifie tes prompts ici
PROMPTS = [
    "Cinematic vertical shot of a rainy city street at night, neon reflections on wet asphalt, slow camera push-in",
    "Close-up of coffee being poured into a cup, warm morning light, shallow depth of field, vertical framing",
    "Aerial view of a foggy mountain range at sunrise, slow drone movement, vertical 9:16",
]
CLIPS_PER_PROMPT = 1
HEIGHT, WIDTH = 480, 832   # portrait 9:16 (multiples de 16)
NUM_FRAMES = 81            # ~5 s a 16 img/s
FPS = 16
GUIDANCE = 5.0
SEED = 42


In [ ]:
# 4) Charger le modele Wan 2.1 (1.3B, Apache 2.0, ~15 Go de telechargement)
import torch
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
pipe = WanPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe.vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)

# Le T4 (16 Go) est juste : on laisse diffusers placer les couches sur le GPU.
pipe.enable_model_cpu_offload()
print("Modele charge.")


In [ ]:
# 5) Generer les clips
from pathlib import Path

OUT = Path("clips_ia"); OUT.mkdir(exist_ok=True)
gen = torch.Generator("cpu").manual_seed(SEED)

for p_index, prompt in enumerate(PROMPTS):
    for k in range(CLIPS_PER_PROMPT):
        print(f"Generation {p_index+1}.{k+1} : {prompt[:60]}...")
        result = pipe(
            prompt=prompt,
            negative_prompt="blurry, low quality, distorted, watermark, text",
            height=HEIGHT, width=WIDTH,
            num_frames=NUM_FRAMES, guidance_scale=GUIDANCE,
            generator=gen,
        ).frames[0]
        path = OUT / f"clip_{p_index:02d}_{k}.mp4"
        export_to_video(result, str(path), fps=FPS)
        print("  ->", path)

print("\nClips generes :")
!ls -lh clips_ia


In [ ]:
# 6) Telecharger l'archive de clips
import shutil
shutil.make_archive("clips_ia", "zip", "clips_ia")
from google.colab import files
files.download("clips_ia.zip")


## Alternatives gratuites

| Modele | Licence | VRAM | Remarque |
|---|---|---|---|
| **Wan 2.1 1.3B** (ce notebook) | Apache 2.0 | ~12 Go | Le plus accessible sur T4 |
| **LTX-Video 2B** | Apache 2.0 | ~12 Go | Beaucoup plus rapide (quasi temps reel) |
| **CogVideoX-2B** | Apache 2.0 | ~12 Go | Bonne qualite, plus lent |

Pour LTX-Video, remplace la cellule 4 par :

```python
from diffusers import LTXPipeline
pipe = LTXPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=torch.bfloat16)
pipe.to("cuda")
```

> La generation video IA demande un GPU : c'est une contrainte materielle, pas une limite
> de ReelForge. Colab fournit ce GPU gratuitement ; ReelForge s'occupe du montage, de la
> voix et des sous-titres.
